In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

base_path = "/content/drive/MyDrive/Prodigy/PetImages"
cat_path = os.path.join(base_path, "Cat")
dog_path = os.path.join(base_path, "Dog")


In [ ]:
img_size = 64

data = []
labels = []

def load_images_from_folder(folder, label):
    for filename in os.listdir(folder):
        file_path = os.path.join(folder, filename)
        try:
            img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, (img_size, img_size))  # minimiz the images
            data.append(img.flatten())   # Flatten -> 4096
            labels.append(label)
        except:
            continue

# Cats =0 , Dogs = 0
load_images_from_folder(cat_path, 0)
load_images_from_folder(dog_path, 1)

print("Dataset size:", len(data))


In [ ]:
X = np.array(data)
y = np.array(labels)

X = X / 255.0

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
svm = SVC(kernel='linear', verbose=True)

svm.fit(X_train, y_train)


In [ ]:
y_pred = svm.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))


In [ ]:
import matplotlib.pyplot as plt

indices = np.random.choice(len(X_test), 10, replace=False)

plt.figure(figsize=(12, 6))
for i, idx in enumerate(indices):
    img = X_test[idx].reshape(64, 64)
    pred = svm.predict([X_test[idx]])[0]
    true = y_test[idx]

    plt.subplot(2, 5, i+1)
    plt.imshow(img, cmap='gray')
    plt.title(f"Pred: {'Dog' if pred==1 else 'Cat'}\nTrue: {'Dog' if true==1 else 'Cat'}")
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Cat", "Dog"],
            yticklabels=["Cat", "Dog"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Cats vs Dogs (SVM)")
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc

y_score = svm.decision_function(X_test)

fpr, tpr, _ = roc_curve(y_test, y_score)

roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, color="blue", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0,1], [0,1], color="red", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Cats vs Dogs (SVM)")
plt.legend(loc="lower right")
plt.show()
